In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nbformat as nbf
import subprocess, sys, textwrap, shutil, zipfile, os

# CUSTOMER CHURN FORECASTING PROJECT

FILE = Path("/mnt/data/Telco-Customer-Churn.csv")
PROJECT = Path("/mnt/data/Customer_Churn_Forecasting_Project")
PROJECT.mkdir(exist_ok=True)

# 1. INSPECT DATA
df_raw = pd.read_csv(FILE)

print("Original dataset shape:", df_raw.shape)
print("\nColumns:")
print(df_raw.columns.tolist())
print("\nMissing values:")
print(df_raw.isna().sum())

# 2. CLEANING
df = df_raw.copy()
df.columns = df.columns.str.strip()

# TotalCharges contains blank strings in the original dataset
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"].astype(str).str.strip(),
    errors="coerce"
)

duplicate_count = df.duplicated().sum()
df = df.drop_duplicates()

missing_totalcharges = df["TotalCharges"].isna().sum()

# Keep customer rows, but impute TotalCharges using MonthlyCharges * tenure
df["TotalCharges"] = df["TotalCharges"].fillna(
    df["MonthlyCharges"] * df["tenure"]
)

# Normalize binary target
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

# Remove impossible negative numeric values if present
df = df[
    (df["tenure"] >= 0) &
    (df["MonthlyCharges"] >= 0) &
    (df["TotalCharges"] >= 0)
].copy()

print("\n========== CLEANING SUMMARY ==========")
print("Original rows:", len(df_raw))
print("Duplicate rows removed:", duplicate_count)
print("Missing TotalCharges before imputation:", missing_totalcharges)
print("Rows after cleaning:", len(df))
print("Churn rate:", round(df["Churn"].mean() * 100, 2), "%")

# 3. FEATURE ENGINEERING
# Tenure-based customer lifecycle feature
df["TenureGroup"] = pd.cut(
    df["tenure"],
    bins=[-1, 6, 12, 24, 48, 72],
    labels=["0-6 months", "7-12 months", "13-24 months",
            "25-48 months", "49-72 months"]
)

# Count subscribed services
service_cols = [
    "PhoneService", "MultipleLines", "OnlineSecurity",
    "OnlineBackup", "DeviceProtection", "TechSupport",
    "StreamingTV", "StreamingMovies"
]

def yes_count(row):
    return sum(
        str(row[c]).strip().lower() == "yes"
        for c in service_cols
        if c in row.index
    )

df["ServiceCount"] = df.apply(yes_count, axis=1)

# Average monthly value over tenure
df["AvgMonthlySpend"] = np.where(
    df["tenure"] > 0,
    df["TotalCharges"] / df["tenure"],
    df["MonthlyCharges"]
)

# Contract / payment behavior features
df["IsMonthToMonth"] = (df["Contract"] == "Month-to-month").astype(int)
df["HasAutoPayment"] = (
    df["PaymentMethod"].isin([
        "Bank transfer (automatic)",
        "Credit card (automatic)"
    ])
).astype(int)

# Internet-service indicator
df["HasInternetService"] = (
    df["InternetService"] != "No"
).astype(int)

print("\nEngineered features:")
print("- TenureGroup")
print("- ServiceCount")
print("- AvgMonthlySpend")
print("- IsMonthToMonth")
print("- HasAutoPayment")
print("- HasInternetService")

# 4. EDA
churn_counts = df["Churn"].value_counts().sort_index()
churn_labels = ["Stayed", "Churned"]

plt.figure(figsize=(7, 5))
plt.bar(churn_labels, [churn_counts.get(0, 0), churn_counts.get(1, 0)])
plt.title("Customer Churn Distribution")
plt.ylabel("Number of Customers")
plt.tight_layout()
plt.savefig(PROJECT / "churn_distribution.png", dpi=150)
plt.show()

# Churn by contract
contract_churn = pd.crosstab(
    df["Contract"], df["Churn"], normalize="index"
) * 100

plt.figure(figsize=(9, 5))
contract_churn[1].sort_values().plot(kind="barh")
plt.title("Churn Rate by Contract Type")
plt.xlabel("Churn Rate (%)")
plt.tight_layout()
plt.savefig(PROJECT / "churn_by_contract.png", dpi=150)
plt.show()

# Churn by tenure group
tenure_churn = pd.crosstab(
    df["TenureGroup"], df["Churn"], normalize="index"
) * 100

plt.figure(figsize=(9, 5))
tenure_churn[1].plot(kind="bar")
plt.title("Churn Rate by Tenure Group")
plt.xlabel("Tenure Group")
plt.ylabel("Churn Rate (%)")
plt.xticks(rotation=25)
plt.tight_layout()
plt.savefig(PROJECT / "churn_by_tenure.png", dpi=150)
plt.show()

# Monthly charges distribution by churn
plt.figure(figsize=(9, 5))
sns.boxplot(data=df, x="Churn", y="MonthlyCharges")
plt.xticks([0, 1], ["Stayed", "Churned"])
plt.title("Monthly Charges by Churn Status")
plt.tight_layout()
plt.savefig(PROJECT / "monthly_charges_by_churn.png", dpi=150)
plt.show()

# Numeric correlations
numeric_cols = [
    "SeniorCitizen", "tenure", "MonthlyCharges",
    "TotalCharges", "ServiceCount", "AvgMonthlySpend", "Churn"
]

plt.figure(figsize=(9, 7))
sns.heatmap(
    df[numeric_cols].corr(),
    annot=True,
    fmt=".2f",
    cmap="Blues"
)
plt.title("Numeric Feature Correlation")
plt.tight_layout()
plt.savefig(PROJECT / "correlation_heatmap.png", dpi=150)
plt.show()

# 5. OUTLIER CHECK
outlier_summary = {}

for col in ["tenure", "MonthlyCharges", "TotalCharges", "AvgMonthlySpend"]:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_summary[col] = int(count)

print("\nIQR outlier counts:")
print(outlier_summary)
print(
    "\nOutliers were inspected rather than blindly deleted because high "
    "customer charges/tenure can be legitimate business observations."
)

# 6. MODEL DATA
target = "Churn"

drop_cols = ["customerID", target]

X = df.drop(columns=drop_cols)
y = df[target]

# Drop TenureGroup from model because the numeric tenure variable already carries
# the underlying information; keeping both can add redundant representation.
X = X.drop(columns=["TenureGroup"])

categorical_features = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numeric_features = X.select_dtypes(
    include=[np.number]
).columns.tolist()

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve
)

# 60% train, 20% validation, 20% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.40,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            numeric_features
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(
                    handle_unknown="ignore",
                    drop="first"
                ))
            ]),
            categorical_features
        )
    ]
)

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=6,
        min_samples_leaf=10,
        class_weight="balanced",
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
}

# Try XGBoost if installed
try:
    from xgboost import XGBClassifier
    scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
    models["XGBoost"] = XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        eval_metric="logloss",
        random_state=42
    )
    xgb_available = True
except Exception as e:
    xgb_available = False
    print("\nXGBoost unavailable; continuing with scikit-learn models.")

# 7. TRAIN + VALIDATE
validation_results = []
fitted_models = {}

for name, model in models.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipe.fit(X_train, y_train)

    val_pred = pipe.predict(X_val)
    val_prob = pipe.predict_proba(X_val)[:, 1]

    validation_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_val, val_pred),
        "Precision": precision_score(y_val, val_pred, zero_division=0),
        "Recall": recall_score(y_val, val_pred, zero_division=0),
        "F1": f1_score(y_val, val_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_val, val_prob)
    })

    fitted_models[name] = pipe

validation_df = pd.DataFrame(validation_results).sort_values(
    "F1", ascending=False
).reset_index(drop=True)

print("\n========== VALIDATION RESULTS ==========")
print(validation_df.round(4))

# Select based on validation F1, because identifying churners is important
best_model_name = validation_df.iloc[0]["Model"]
best_model = fitted_models[best_model_name]

# 8. FINAL TEST EVALUATION
test_pred = best_model.predict(X_test)
test_prob = best_model.predict_proba(X_test)[:, 1]

test_metrics = {
    "Accuracy": accuracy_score(y_test, test_pred),
    "Precision": precision_score(y_test, test_pred, zero_division=0),
    "Recall": recall_score(y_test, test_pred, zero_division=0),
    "F1": f1_score(y_test, test_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, test_prob)
}

print("\n========== FINAL TEST RESULTS ==========")
for metric, value in test_metrics.items():
    print(f"{metric}: {value:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        test_pred,
        target_names=["Stayed", "Churned"],
        zero_division=0
    )
)

# Confusion matrix
cm = confusion_matrix(y_test, test_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Stayed", "Churned"],
    yticklabels=["Stayed", "Churned"]
)
plt.title(f"Confusion Matrix - {best_model_name}")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.savefig(PROJECT / "confusion_matrix.png", dpi=150)
plt.show()

# 9. MODEL COMPARISON
metric_cols = ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]

validation_df.set_index("Model")[metric_cols].plot(
    kind="bar",
    figsize=(12, 6)
)
plt.title("Model Performance Comparison - Validation Set")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=20)
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig(PROJECT / "model_performance_comparison.png", dpi=150)
plt.show()

# ROC curves
plt.figure(figsize=(9, 7))

for name, pipe in fitted_models.items():
    prob = pipe.predict_proba(X_val)[:, 1]
    fpr, tpr, _ = roc_curve(y_val, prob)
    auc = roc_auc_score(y_val, prob)
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--")
plt.title("ROC Curves - Validation Set")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.tight_layout()
plt.savefig(PROJECT / "roc_curves.png", dpi=150)
plt.show()

# 10. FEATURE IMPORTANCE
# Obtain transformed feature names
fitted_preprocessor = best_model.named_steps["preprocessor"]
model_obj = best_model.named_steps["model"]

feature_names = fitted_preprocessor.get_feature_names_out()

if hasattr(model_obj, "feature_importances_"):
    importances = model_obj.feature_importances_
else:
    importances = np.abs(model_obj.coef_[0])

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values("Importance", ascending=False)

top_features = feature_importance.head(15).sort_values("Importance")

plt.figure(figsize=(10, 7))
plt.barh(top_features["Feature"], top_features["Importance"])
plt.title(f"Top Churn Predictors - {best_model_name}")
plt.xlabel("Importance")
plt.tight_layout()
plt.savefig(PROJECT / "feature_importance.png", dpi=150)
plt.show()

print("\n TOP CHURN PREDICTORS")
print(feature_importance.head(15).to_string(index=False))

# -----------------------------
# 11. BUSINESS INSIGHTS
# -----------------------------
churn_rate = df["Churn"].mean() * 100
month_to_month_churn = (
    df.loc[df["Contract"] == "Month-to-month", "Churn"].mean() * 100
)
one_year_churn = (
    df.loc[df["Contract"] == "One year", "Churn"].mean() * 100
)
two_year_churn = (
    df.loc[df["Contract"] == "Two year", "Churn"].mean() * 100
)
auto_pay_churn = (
    df.loc[df["HasAutoPayment"] == 1, "Churn"].mean() * 100
)
non_auto_pay_churn = (
    df.loc[df["HasAutoPayment"] == 0, "Churn"].mean() * 100
)

print("\n" + "=" * 60)
print("KEY BUSINESS INSIGHTS")
print("=" * 60)
print(f"Overall churn rate: {churn_rate:.2f}%")
print(f"Month-to-month contract churn: {month_to_month_churn:.2f}%")
print(f"One-year contract churn: {one_year_churn:.2f}%")
print(f"Two-year contract churn: {two_year_churn:.2f}%")
print(f"Automatic-payment churn: {auto_pay_churn:.2f}%")
print(f"Non-automatic-payment churn: {non_auto_pay_churn:.2f}%")

# 12. SAVE OUTPUTS

validation_df.to_csv(PROJECT / "model_validation_results.csv", index=False)
pd.DataFrame([test_metrics]).to_csv(
    PROJECT / "final_test_metrics.csv", index=False
)
feature_importance.to_csv(
    PROJECT / "feature_importance.csv", index=False
)

cleaned_output = df.copy()
cleaned_output.to_csv(
    PROJECT / "cleaned_telco_churn.csv",
    index=False
)

summary = f"""
CUSTOMER CHURN FORECASTING PROJECT
===================================

Original rows: {len(df_raw):,}
Cleaned rows: {len(df):,}
Original columns: {df_raw.shape[1]}
Churn rate: {churn_rate:.2f}%

Train rows: {len(X_train):,}
Validation rows: {len(X_val):,}
Test rows: {len(X_test):,}

Selected model: {best_model_name}

Final test metrics:
Accuracy: {test_metrics["Accuracy"]:.4f}
Precision: {test_metrics["Precision"]:.4f}
Recall: {test_metrics["Recall"]:.4f}
F1: {test_metrics["F1"]:.4f}
ROC-AUC: {test_metrics["ROC-AUC"]:.4f}

Contract churn:
Month-to-month: {month_to_month_churn:.2f}%
One year: {one_year_churn:.2f}%
Two year: {two_year_churn:.2f}%

Payment behavior:
Automatic payment: {auto_pay_churn:.2f}%
Non-automatic payment: {non_auto_pay_churn:.2f}%

Top churn predictors:
{feature_importance.head(10).to_string(index=False)}
"""

(PROJECT / "project_summary.txt").write_text(summary, encoding="utf-8")

print("\nPROJECT ANALYSIS COMPLETE")
print("Output folder:", PROJECT)